In [3]:
import requests
import pandas as pd
from datetime import datetime

# --- 설정 ---
# 분석하고 싶은 선수의 ID와 이름을 설정합니다.
# (ID는 Transfermarkt URL에서 확인할 수 있습니다: .../spieler/866246)
player_id = "355915"
player_name = "Cristian Romero"

# --- 데이터 요청 ---
# 1. 데이터 API의 URL 주소입니다. {player_id} 부분에 위에서 설정한 선수 ID가 들어갑니다.
api_url = f"https://www.transfermarkt.com/ceapi/marketValueDevelopment/graph/{player_id}"

# 2. 서버가 봇으로 인식하지 않도록 실제 브라우저 정보(User-Agent)를 헤더에 포함시킵니다.
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
}

print(f"'{player_name}' 선수의 몸값 변화 데이터를 가져옵니다...")

try:
    # 3. requests 라이브러리를 사용해 API에 GET 요청을 보냅니다.
    response = requests.get(api_url, headers=headers)
    response.raise_for_status()  # 요청이 실패하면 여기서 에러가 발생합니다.

    # 4. 서버로부터 받은 응답을 JSON 형태로 변환합니다.
    json_data = response.json()
    
    # --- 데이터 처리 및 변환 ---
    # 5. JSON 데이터에서 실제 몸값 기록이 담긴 'list' 부분을 추출합니다.
    market_value_list = json_data.get('list', [])

    # 6. 추출한 데이터를 파이썬 딕셔너리 리스트로 재구성합니다.
    processed_data = []
    for item in market_value_list:
        processed_data.append({
            '날짜': datetime.fromtimestamp(item['x'] / 1000).strftime('%Y-%m-%d'),
            '시장 가치 (EUR)': item['y'],
            '소속팀': item['verein'],
            '나이': item['age']
        })
        
    # 7. 재구성한 데이터를 pandas의 DataFrame(표)으로 변환합니다.
    df = pd.DataFrame(processed_data)

    # 8. 시장 가치 컬럼을 사람이 보기 편한 형태로 변환합니다. (예: 1000000 -> 1.00m)
    def format_value(value):
        if value >= 1000000:
            return f"€{value / 1000000:.2f}m"
        elif value >= 1000:
            return f"€{value / 1000:.0f}k"
        return f"€{value}"

    df['시장 가치 (표기)'] = df['시장 가치 (EUR)'].apply(format_value)
    
    # --- 결과 출력 ---
    print("\n[ 데이터 분석 결과 ]")
    
    # 9. 최종 결과를 출력합니다.
    # to_string()을 사용하면 모든 행이 잘리지 않고 출력됩니다.
    print(df[['날짜', '소속팀', '나이', '시장 가치 (표기)']].to_string(index=False))

except requests.exceptions.RequestException as e:
    print(f"오류: 데이터를 가져오는 데 실패했습니다. (네트워크 오류) - {e}")
except Exception as e:
    print(f"오류: 데이터를 처리하는 중 문제가 발생했습니다 - {e}")

'Cristian Romero' 선수의 몸값 변화 데이터를 가져옵니다...

[ 데이터 분석 결과 ]
        날짜                    소속팀 나이 시장 가치 (표기)
2016-02-06         CA Belgrano II 17      €150k
2017-02-13 Club Atlético Belgrano 18     €2.00m
2017-10-09 Club Atlético Belgrano 19     €1.00m
2018-05-08 Club Atlético Belgrano 20      €750k
2018-12-20              Genoa CFC 20     €4.00m
2019-03-01              Genoa CFC 20    €15.00m
2019-06-06              Genoa CFC 21    €15.00m
2019-12-12              Genoa CFC 21    €15.00m
2020-04-08              Genoa CFC 21    €13.50m
2020-08-25              Genoa CFC 22    €14.00m
2020-12-29            Atalanta BC 22    €15.00m
2021-03-23            Atalanta BC 22    €25.00m
2021-06-03            Atalanta BC 23    €30.00m
2021-07-16            Atalanta BC 23    €35.00m
2021-12-23      Tottenham Hotspur 23    €35.00m
2022-03-29      Tottenham Hotspur 23    €40.00m
2022-06-15      Tottenham Hotspur 24    €48.00m
2022-09-15      Tottenham Hotspur 24    €55.00m
2022-11-03      Tottenham Hotsp